In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import storage,bigquery

In [ ]:
!gcloud projects list

In [ ]:
client = storage.Client(project='sales-data-506808')

In [ ]:
buckets = client.list_buckets()

print("Buckets in your project:")
for bucket in buckets:
    print(f"- {bucket.name}")


In [ ]:
bucket = client.get_bucket("store-sales-raw-data")

In [ ]:
# Creating new folder

blob = bucket.blob("sales/")
blob.upload_from_string("")

print("Created folder 'sales/' successfully")

In [ ]:
# Uploading file to csv

blob = bucket.blob("sales/retail_sales.csv")
blob.upload_from_filename("/content/retail_sales.csv")

print("File uploaded sucessfully")

In [ ]:
# Reading the retail_sales file

blob = bucket.blob("sales/retail_sales.csv")
blob.download_to_filename("retail_sales.csv")

In [ ]:
df = pd.read_csv("retail_sales.csv")

In [ ]:
df

In [ ]:
df.info()

In [ ]:
columns= ['Transaction_ID', 'Customer_ID', 'Category', 'Item', 'Price_Per_Unit',
       'Quantity', 'Total_Spent', 'Payment_Method', 'order_placement',
       'Transaction_Date', 'Discount_Applied']
columns = [c.lower() for c in columns]

In [ ]:
df.columns = columns

In [ ]:
df.isnull().sum()

In [ ]:
# Convert columns to appropriate data types

df['transaction_id'] = df['transaction_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)

df['price_per_unit'] = pd.to_numeric(df['price_per_unit'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce')

df['transaction_date'] = pd.to_datetime(
    df['transaction_date'],
    errors='coerce'
)

df['discount_applied'] = df['discount_applied'].astype(bool)

In [ ]:
# Check unique categories

print("Categories:")
print(df['category'].unique())

In [ ]:
#Fill missing discount_applied values

df['discount_applied'] = df['discount_applied'].fillna(False,inplace=False)

In [ ]:
#Filling missing item values

item_map = df.dropna(subset=['item']).groupby('price_per_unit')['item'].first()
df['item'] = df['item'].fillna(df['price_per_unit'].map(item_map))

In [ ]:
#Filling missing price_per_unit values


def fillingmissingvalues(df,target,formula):
  mask = df[target].isna()
  df.loc[mask, target] = formula(df.loc[mask])
  return df

fillingmissingvalues(df,"price_per_unit", lambda sub: sub['total_spent']/sub['quantity'])

In [ ]:
#Filling missing quantity values

fillingmissingvalues(df,"quantity", lambda sub: sub['total_spent']/sub['price_per_unit'])

In [ ]:
#Filling missing total values

fillingmissingvalues(df,"total_spent", lambda sub: sub['quantity']*sub['price_per_unit'])

In [ ]:
df[df['quantity'].isna() & df['total_spent'].isna()]

In [ ]:
#Droping rows that has missing values in both quantity and total spent

df = df.dropna(subset=['quantity','total_spent'], how='all')

In [ ]:
# Overall sales summary

total_revenue = df['total_spent'].sum()
total_volume = df['quantity'].sum()
total_transactions = df['transaction_id'].nunique()
total_customers = df['customer_id'].nunique()

print("Total Revenue:", round(total_revenue, 2))
print("Total Volume:", total_volume)
print("Total Transactions:", total_transactions)
print("Total Customers:", total_customers)

In [ ]:
df.to_csv("retail_sales_cleaned.csv", index=False)

In [ ]:
blob = bucket.blob('sales/retail_sales_cleaned.csv')
blob.upload_from_filename('retail_sales_cleaned.csv')